# ViT runner for Google Colab

This notebook is designed to run from outside the `VAQ-apt` folder.
It will try to locate the repository automatically by searching parent folders for `src/models/patch_embed.py`.


In [ ]:
!pip install -U pip
!pip install torch torchvision timm numpy einops ipdb


In [ ]:
from pathlib import Path
import os
import sys

def find_repo(start: Path):
    for p in [start, *start.parents]:
        candidate = p / "VAQ-apt" / "src" / "models" / "patch_embed.py"
        if candidate.exists():
            return candidate.parent.parent.parent
        candidate2 = p / "src" / "models" / "patch_embed.py"
        if candidate2.exists():
            return p
    return None

# Try to find the repo automatically.
repo_dir = find_repo(Path.cwd())

# Fallback for Colab if the repo was cloned to /content/VAQ-apt.
if repo_dir is None:
    fallback = Path('/content/VAQ-apt')
    if fallback.exists():
        repo_dir = fallback
    else:
        # If the repo is somewhere else, set this manually.
        repo_dir = Path('/absolute/path/to/VAQ-apt')

repo_dir = repo_dir.resolve()
if not repo_dir.exists():
    raise FileNotFoundError(f"Could not find the repository at {repo_dir}. Set the path manually.")

os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))

print('Repository directory:', repo_dir)


In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU device:', torch.cuda.get_device_name(0))


In [ ]:
import torch
from src.models.patch_embed import PatchEmbed

model = PatchEmbed(img_size=224, patch_size=16, in_chans=3, embed_dim=768).eval()
x = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    y = model(x)

print('input shape:', x.shape)
print('output shape:', y.shape)


In [ ]:
# Search for the full ViT entrypoint in src/models/
!grep -R "class .*ViT\|class .*VisionTransformer\|class .*Model" -n /content/VAQ-apt/src/models 2>/dev/null || true


## Notes

- If the repository was not found automatically, edit the fallback path in the second cell to point to your local/Colab repo path.
- If you see import errors, rerun the second cell so the repo is added to `sys.path`.
- If you hit memory errors, reduce the batch size when running any training/evaluation script.
